# 🚀 Why LangChain? From Raw Text to Reliable Data

Welcome to this hands-on session! Today, we are exploring why developers use orchestration frameworks like **LangChain** instead of writing raw API calls.

### 🎯 The Session Goal
We want to take a dynamic user input (e.g., a product category), ask an LLM for a list of items, and cleanly parse that output into a **usable Python list** for our application backend.

### 🧠 The Core Conflict: Manual vs. Framework
1. **The Manual Way (OpenAI API Only)**: We use manual string building (`f-strings`) and manual parsing (`json.loads`). It is quick to set up but incredibly fragile in production when LLMs return unpredictable markdown formatting.
2. **The LangChain Way**: We use `PromptTemplates` and `OutputParsers` chained together cleanly via LangChain Expression Language (LCEL). This makes our code modular, reusable, and resilient to LLM syntax changes.

---
*Ensure you have your `OPENAI_API_KEY` ready in your Colab Secrets (the 🔑 icon on the left sidebar) under the name `OPENAI_API_KEY` before running the cells below.*


In [1]:
# Install the required packages
!pip install -q langchain-core langchain-openai openai

import os
from google.colab import userdata

# Securely set your OpenAI API key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 24.8 MB/s eta 0:00:00


## 📑 Section 1: Manual Orchestration (The OpenAI-Only Approach)

In this section, we build everything by hand. We use a standard Python `f-string` to inject our variable and try to "beg" the LLM in plain text to give us clean JSON.

### Look out for:
* **The String Mess**: Look at how messy the prompt engineering text becomes.
* **The Fragility**: We rely entirely on `json.loads()`. If the LLM throws in a single extra conversational word, the whole application crashes.


In [2]:
import openai
import json

client = openai.OpenAI()
category = "smartphones"

# 1. Manual prompt stuffing (Trying to force JSON format)
manual_prompt = f"""List 3 popular items in the {category} category.
Return ONLY a valid JSON array of strings.
Do not include conversational text or markdown formatting like ```json."""

print("--- Sending Manual Prompt ---")
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": manual_prompt}],
    temperature=0.2 # Higher temperature to show unpredictability
)

raw_output = response.choices[0].message.content
print(f"Raw LLM Output:\n{raw_output}\n")
print(f"Raw Output Type: {type(raw_output)}")

# 2. Manual parsing (Fragile and error-prone)
print("--- Parsing Manual Output ---")
try:
    parsed_list = json.loads(raw_output)
    print(f"Successfully parsed Python List: {parsed_list}")
except json.JSONDecodeError as e:
    print(f"❌ CRASHED! JSON Parsing Failed: {e}")
    print("Why? The LLM likely included markdown fences (```json) or introductory text.")


--- Sending Manual Prompt ---
Raw LLM Output:
["iPhone 14 Pro", "Samsung Galaxy S23", "Google Pixel 7"]

Raw Output Type: <class 'str'>
--- Parsing Manual Output ---
Successfully parsed Python List: ['iPhone 14 Pro', 'Samsung Galaxy S23', 'Google Pixel 7']


---

## 📑 Section 2: The LangChain Way (The Production-Ready Solution)

Now, let's look at how LangChain solves this problem by separating our application logic from the LLM formatting instructions.

Instead of manual strings, we use **PromptTemplates** and **OutputParsers** glued together with **LangChain Expression Language (LCEL)** using the pipe (`|`) operator.

### Why this is better:
1. **Automated Instructions**: `parser.get_format_instructions()` automatically injects strict, programmatic formatting tokens into the prompt behind the scenes.
2. **Type Safety**: The `.invoke()` method directly outputs a natively parsed Python object. No `json.loads()` required.
3. **Modularity**: If we want a comma-separated list or a Pydantic object tomorrow, we only change the parser object, not our prompt text.


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI

# 1. Initialize the components
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
parser = JsonOutputParser()

# 2. Let the parser inject its own bulletproof instructions
format_instructions = parser.get_format_instructions()

# 3. Create a reusable template
langchain_prompt = PromptTemplate(
    template="List 3 popular items in the {category} category.\n{format_instructions}",
    input_variables=["category"],
    partial_variables={"format_instructions": format_instructions}
)

# 4. Seamlessly chain them together using LCEL (|)
chain = langchain_prompt | model | parser

print("--- Sending LangChain Request ---")
# Invoke returns a perfectly parsed Python data structure automatically
langchain_output = chain.invoke({"category": "smartphones"})

print(f"Parsed Output: {langchain_output}")
print(f"Parsed Output Type: {type(langchain_output)}")


## 🏁 Conclusion: The Big Takeaway

We saw two completely different ways to handle LLM inputs and outputs:

1. **Manual Orchestration** forced us to write fragile string formatting rules, manually handle nested API lists (`choices[0]`), and risk application crashes if the LLM returned unpredictable text wrappers.
2. **LangChain** abstracted all of that structural complexity away. By combining `PromptTemplates` and `OutputParsers` into a unified pipeline (`|`), we achieved clean, robust, and type-safe data parsing.

### 🚀 When to use LangChain in your projects:
* When you need your AI output to reliably feed into a database, frontend UI, or external API.
* When your prompts start growing in complexity and require dynamic variables or multiple steps.
* When you want the flexibility to switch from OpenAI to another model provider (like Anthropic or Google Gemini) later without rewriting your entire parsing logic.

---